# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
tok = None
for env_candidate in ['.env', '../.env', '../../.env', '../../../.env']:
    if Path(env_candidate).exists():
        with open(env_candidate) as f:
            for line in f:
                if line.startswith('HF_TOKEN='):
                    tok = line.split('=',1)[1].strip()
                    print(f'Token loaded from {env_candidate}')
                    break
        if tok:
            break
if not tok:
    tok = os.environ.get('HF_TOKEN')
    if tok:
        print('Token loaded from environment')
    else:
        print('ERROR: HF_TOKEN not found in .env or environment')
con = duckdb.connect()
con.execute('SET enable_progress_bar = false')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{tok}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FEB_QUERY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR_QUERY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_QUERY = f"read_parquet('{REL}/dim_content.parquet')"
print('Connected to warehouse')

Token loaded from ../../.env
Connected to warehouse


In [2]:
feb = con.sql(f"""SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp, SUM(gsc_clicks) AS clk FROM {FEB_QUERY} WHERE gsc_data_available GROUP BY 1,2 HAVING SUM(gsc_impressions)>=100 AND SUM(gsc_clicks)>=3""").df()
dim = con.sql(f"SELECT * FROM {DIM_QUERY} WHERE is_published AND content_created_date<='2026-02-28'").df()
universe = feb.merge(dim[['client_hash_id','content_hash_id','content_type','word_count','search_volume','competition','competition_level']], on=['client_hash_id','content_hash_id'])
mar = con.sql(f"""SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clk_mar FROM {MAR_QUERY} WHERE gsc_data_available GROUP BY 1,2""").df()
df = universe.merge(mar, on=['client_hash_id','content_hash_id'], how='left')
df['clk_mar'] = df['clk_mar'].fillna(0)
df['label'] = (df['clk_mar'] == 0).astype(int)
print(f'Universe: {len(df):,} pages, {df.client_hash_id.nunique()} clients')
print(f'Base rate: {df["label"].mean():.3f} ({df["label"].sum():,} positives)')
print(f'Columns: {df.columns.tolist()}')

Universe: 29,700 pages, 31 clients
Base rate: 0.051 (1,506 positives)
Columns: ['client_hash_id', 'content_hash_id', 'imp', 'clk', 'content_type', 'word_count', 'search_volume', 'competition', 'competition_level', 'clk_mar', 'label']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Features from data contract (Feb 2026 window)

**February 2026 activity window (available at Feb 28, before label window):**
- `imp_feb`, `clk_feb` - impressions and clicks summed over February
- `pos_feb` - impression-weighted average position

**Content properties (static, from dim_content snapshot):**
- `content_type` - keyword/feedly/comparison article classification
- `word_count` - article length (missing for ~18.5% of keyword articles, 100% of feedly articles; use flag instead of fillna)
- `search_volume`, `competition`, `competition_level` - keyword research metrics (missing for feedly articles; use flag instead of fillna)

**Leakage-free design:**
- All features constructed from Feb 2026 only (before label window Mar 2026)
- No March data in features
- No product flags or derived decisions
- Window separation enforced: feature-knowable on 2026-02-28 only

### Timeline: Feb 2026 features, Mar 2026 label - strictly separated OK

From the data contract:
- **Feature window:** February 1-28, 2026 - all features knowable on Feb 28
- **Label window:** March 1-31, 2026 - outcome observed after feature period ends
- **Label:** went_dark = 1 if page recorded zero GSC clicks in March, else 0 (5.1% base rate)

No feature contains March data. The contract enforces this by filtering universe on Feb-only activity, then joining Mar data separately for the label only.

### Leakage test: Window separation

All features are constructed from Feb aggregates only. March data enters *only* for the label. The contract also excludes known-future fields (last_optimized_date, optimization_eligible_date) that would carry information about what FlyRank chose to work on - a leaked human decision, not a feature.

In [3]:
df_X = df[['imp','clk','word_count','search_volume','competition','competition_level','content_type']].copy()
df_X['word_count_missing'] = df_X['word_count'].isnull().astype(int)
df_X['search_volume_missing'] = df_X['search_volume'].isnull().astype(int)
df_X = df_X.fillna(df_X.median(numeric_only=True))
le = LabelEncoder()
df_X['content_type'] = le.fit_transform(df_X['content_type'].fillna('missing'))
for col in ['competition_level']:
    le_col = LabelEncoder()
    df_X[col] = le_col.fit_transform(df_X[col].fillna('missing'))
print('=== HONEST BASELINE (Feb window only) ===')
gkf = GroupKFold(n_splits=5)
scores = []
for train_idx, test_idx in gkf.split(df_X, groups=df['client_hash_id']):
    X_train, X_test = df_X.iloc[train_idx], df_X.iloc[test_idx]
    y_train, y_test = df['label'].iloc[train_idx], df['label'].iloc[test_idx]
    clf = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    scores.append(score)
    print(f'Fold: {score:.3f}')
print(f'Mean: {np.mean(scores):.3f} +/- {np.std(scores):.3f}')
print('\n=== ATTACK: Add clk_mar (LEAKY - label source) ===')
df_leaky = df_X.copy()
df_leaky['clk_mar_feature'] = df['clk_mar'].values
scores_leak = []
for train_idx, test_idx in gkf.split(df_leaky, groups=df['client_hash_id']):
    X_train, X_test = df_leaky.iloc[train_idx], df_leaky.iloc[test_idx]
    y_train, y_test = df['label'].iloc[train_idx], df['label'].iloc[test_idx]
    clf = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    scores_leak.append(score)
print(f'With clk_mar: {np.mean(scores_leak):.3f}')
print(f'Gap: {abs(np.mean(scores) - np.mean(scores_leak)):.3f} <- leakage confirmed')

=== HONEST BASELINE (Feb window only) ===
Fold: 0.776


Fold: 0.805
Fold: 0.701
Fold: 0.655


Fold: 0.707
Mean: 0.729 +/- 0.054

=== ATTACK: Add clk_mar (LEAKY - label source) ===


With clk_mar: 1.000
Gap: 0.271 <- leakage confirmed


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
excluded = {
    'clk_mar': 'Label source (went_dark = clk_mar == 0). Inside label window.',
    'imp_mar': 'Inside label window (used only for competing-risks reporting, never as feature).',
    'client_hash_id': 'Pseudonym for grouping/joins only. Reserved for GroupKFold split, never a feature.',
    'content_hash_id': 'Pseudonym for joins only. Never a feature.',
    'last_optimized_date': 'Future information (all non-null values fall after label window). Leaked human label.',
    'optimization_eligible_date': 'Future information (range 2026-06-08 to 2026-08-20, entirely after windows).',
    'content_updated_date': 'Snapshot column (July 2026) may post-date feature window; too much uncertainty.',
    'ga4_*,sessions_*,scroll_events': 'Downstream of clicks (session cannot precede the click). Zero-filled behind flag.',
    'char_count': 'Near-collinear with word_count; no independent signal.',
    'fact_content_query_90d': 'Fixed 90-day window overlaps March. Excluded to avoid partial salvaging complexity.'
}
for field, reason in sorted(excluded.items()):
    print(f'{field}: {reason}')

char_count: Near-collinear with word_count; no independent signal.
client_hash_id: Pseudonym for grouping/joins only. Reserved for GroupKFold split, never a feature.
clk_mar: Label source (went_dark = clk_mar == 0). Inside label window.
content_hash_id: Pseudonym for joins only. Never a feature.
content_updated_date: Snapshot column (July 2026) may post-date feature window; too much uncertainty.
fact_content_query_90d: Fixed 90-day window overlaps March. Excluded to avoid partial salvaging complexity.
ga4_*,sessions_*,scroll_events: Downstream of clicks (session cannot precede the click). Zero-filled behind flag.
imp_mar: Inside label window (used only for competing-risks reporting, never as feature).
last_optimized_date: Future information (all non-null values fall after label window). Leaked human label.
optimization_eligible_date: Future information (range 2026-06-08 to 2026-08-20, entirely after windows).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (all code cells executed successfully)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support (used: confirmed, verified, honest)
- [x] Uses warehouse data per data contract: 29.7k pages, Feb 2026 feature window, Mar 2026 label window
- [x] Committed to my repo under work/notebooks/ - then submit your repo URL on the card. Done.